In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import re
import pyarrow

In [2]:
features_df = pd.read_parquet(r"C:\Users\javie\OneDrive - INSTITUTO TECNOLOGICO AUTONOMO DE MEXICO\MaestriaEnCienciaDeDatos\EstanciaDeInvestigacion\Popocatepelt\PopocatepetlVolcano\data\features\features.parquet", engine="pyarrow")

In [ ]:
features_df = features_df.drop(columns=['frecuency_index'])

In [3]:
features_df = features_df[(features_df['energy']>=0)&(features_df['kurtosis']<2500)&(features_df['frequency_index']>=-3)&(features_df['entropy']>=0)&(features_df['entropy']<15000)]

In [10]:
features_df.head(5)

,timestamp,energy,kurtosis,entropy,frequency_index
0,2022-01-01 00:05:00,1.186654e+07,4.305041,6743.988806,-0.005959
1,2022-01-01 00:10:00,1.047550e+07,4.293156,7043.316623,-0.109812
2,2022-01-01 00:15:00,1.026374e+07,5.644546,7175.473832,0.032564
3,2022-01-01 00:20:00,1.024407e+07,5.625740,7186.277967,0.065660
4,2022-01-01 00:25:00,9.691936e+06,5.402203,7290.097859,-0.179743


In [9]:
features_df.columns

Index(['timestamp', 'energy', 'kurtosis', 'entropy', 'frequency_index'], dtype='object')

In [11]:
features_df['frequency_index']

0        -0.005959
1        -0.109812
2         0.032564
3         0.065660
4        -0.179743
            ...   
244232    0.013671
244233    0.043806
244234    0.213128
244235    0.218329
244236    0.158764
Name: frequency_index, Length: 240025, dtype: float64

In [ ]:
# List of columns to calculate autocorrelation for
columns_to_autocorrelate = ['energy', 'kurtosis', 'entropy', 'frequency_index']

In [ ]:
# Function to calculate autocorrelation for a range of lags
def find_lag_closest_to_zero(series, max_lag=11*96):
    autocorr_values = [series.autocorr(lag) if series.autocorr(lag) > 1e-6 else 0 for lag in range(11, max_lag + 1, 11)]
    closest_lag = np.argmin(np.abs(autocorr_values)) + 1  # Add 1 because lags start at 1
    return closest_lag, autocorr_values[closest_lag - 1]

In [25]:
# Find the lag closest to zero for each column
lags_closest_to_zero = {}
for column in columns_to_autocorrelate:
    lag, autocorr_value = find_lag_closest_to_zero(features_df[column], max_lag = 11*720)
    lags_closest_to_zero[column] = (lag, autocorr_value)

# Display the results
for column, (lag, autocorr_value) in lags_closest_to_zero.items():
    print(f"Column: {column}, Lag closest to zero: {lag}, Autocorrelation: {autocorr_value}")

Column: energy, Lag closest to zero: 202, Autocorrelation: 0
Column: kurtosis, Lag closest to zero: 461, Autocorrelation: 0.0316917955418929
Column: entropy, Lag closest to zero: 717, Autocorrelation: 0.3320062478089116
Column: frequency_index, Lag closest to zero: 688, Autocorrelation: 0.11787346342642852
